In [1]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 128.8 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [ ]:
# ============================================================================
# Social IQa — Social Commonsense Reasoning — Kaggle Benchmark Task
# ============================================================================
# Task: Given a social situation (context) and a question about social
#        implications, select the correct answer from 3 choices (A/B/C).
#        TEXT input ONLY — no images.
# Input: Context + question + 3 answer choices
# Output: One of 3 answer labels (A, B, C)
# Metric: Weighted-Average F1 Score (float, 0.0 - 1.0)
# Samples: 250 (deterministic, seed=42, from validation set)
# ============================================================================

import kaggle_benchmarks as kbench
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import numpy as np
import os
import re

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/kaggle/input/datasets/thedevastator/social-i-qa-a-dataset-for-social-inquiry-questio"
NUM_SAMPLES = 250
RANDOM_SEED = 42

VALID_ANSWERS = ["A", "B", "C"]

LABEL_MAP = {"1": "A", "2": "B", "3": "C"}

ANSWER_SYNONYMS = {
    "a": "A", "b": "B", "c": "C",
    "answer a": "A", "answer b": "B", "answer c": "C",
    "option a": "A", "option b": "B", "option c": "C",
    "answera": "A", "answerb": "B", "answerc": "C",
    "1": "A", "2": "B", "3": "C",
    "answer 1": "A", "answer 2": "B", "answer 3": "C",
    "option 1": "A", "option 2": "B", "option 3": "C",
    "first": "A", "second": "B", "third": "C",
}


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_validation_csv(dataset_root):
    """Locate the validation CSV file."""
    for dirpath, _, filenames in os.walk(dataset_root):
        for fname in filenames:
            fl = fname.lower()
            if "validation" in fl and fl.endswith(".csv"):
                return os.path.join(dirpath, fname)
            if "val" in fl and fl.endswith(".csv") and "train" not in fl:
                return os.path.join(dirpath, fname)
    raise FileNotFoundError(
        f"Could not find validation CSV under {dataset_root}"
    )


def load_and_prepare(csv_path):
    """Load CSV, map labels, drop invalid rows."""
    df = pd.read_csv(csv_path)

    required = ["context", "question", "answerA", "answerB", "answerC", "label"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Convert label to string for consistent mapping
    df["label"] = df["label"].astype(str).str.strip()

    # Map "1"→"A", "2"→"B", "3"→"C"
    df["answer"] = df["label"].map(LABEL_MAP)

    # Drop rows with unmapped labels
    before = len(df)
    df = df.dropna(subset=["answer"]).reset_index(drop=True)
    after = len(df)
    if before != after:
        print(f"[INFO] Dropped {before - after} rows with invalid labels.")

    print(f"[INFO] Loaded {len(df)} valid samples from {csv_path}")
    print(f"[INFO] Answer distribution:")
    print(df["answer"].value_counts().sort_index().to_string())
    return df


def select_samples(df, n=NUM_SAMPLES, seed=RANDOM_SEED):
    """Select N samples with fixed random seed."""
    actual_n = min(n, len(df))
    selected = df.sample(n=actual_n, random_state=seed).reset_index(drop=True)
    print(f"[INFO] Selected {len(selected)} samples (seed={seed})")
    print(f"[INFO] Selection distribution:")
    print(selected["answer"].value_counts().sort_index().to_string())
    return selected


def build_prompt(context, question, answer_a, answer_b, answer_c):
    """Build the multiple-choice prompt."""
    return (
        "You are an expert in social commonsense reasoning.\n\n"
        "Read the following social situation and question, then select "
        "the most appropriate answer.\n\n"
        f"Context: \"{context}\"\n\n"
        f"Question: \"{question}\"\n\n"
        f"A) {answer_a}\n"
        f"B) {answer_b}\n"
        f"C) {answer_c}\n\n"
        "Which answer is correct? Respond with ONLY a single letter: A, B, or C."
    )


def normalize_answer(raw_response):
    """Normalize the LLM's response to A, B, or C."""
    if raw_response is None:
        return "INVALID"

    text = str(raw_response).strip()

    # Check if the first non-whitespace character is A, B, or C
    first_char = text.upper()[:1]
    if first_char in VALID_ANSWERS:
        return first_char

    # Clean and check synonyms
    cleaned = re.sub(r"[^a-z0-9 ]", "", text.lower()).strip()

    if cleaned in ANSWER_SYNONYMS:
        return ANSWER_SYNONYMS[cleaned]

    # Look for patterns like "The answer is A" or "A)"
    for letter in VALID_ANSWERS:
        patterns = [
            rf"\b{letter}\b",
            rf"\b{letter}\)",
            rf"answer\s*(?:is\s*)?{letter}\b",
            rf"option\s*{letter}\b",
        ]
        for pat in patterns:
            if re.search(pat, text, re.IGNORECASE):
                return letter

    # Last resort: check if any valid answer appears
    for letter in VALID_ANSWERS:
        if letter in text.upper().split():
            return letter

    return "INVALID"


# ============================================================================
# MAIN BENCHMARK TASK
# ============================================================================

@kbench.task(
    name="social_iqa_reasoning",
    description=(
        "Social commonsense reasoning on the Social IQa dataset. "
        "The model receives a social situation and a question about "
        "social implications, then selects the correct answer from "
        "3 choices. Evaluated with Weighted-Average F1 Score."
    ),
)
def social_iqa_reasoning(llm) -> float:
    """
    Evaluate an LLM's social commonsense reasoning ability.
    Returns Weighted-Average F1 score between 0.0 and 1.0.
    """

    # STEP 1: Locate and load data
    csv_path = find_validation_csv(DATASET_ROOT)
    df = load_and_prepare(csv_path)

    # STEP 2: Select samples
    samples = select_samples(df, n=NUM_SAMPLES)

    # STEP 3: Run predictions
    y_true = []
    y_pred = []
    results_log = []
    total = len(samples)

    for i, (_, row) in enumerate(samples.iterrows()):
        ground_truth = row["answer"]
        context = row["context"]
        question = row["question"]

        display_ctx = context[:70] + "..." if len(context) > 70 else context
        print(f"\n[SAMPLE {i+1}/{total}]")
        print(f"  Context:  \"{display_ctx}\"")
        print(f"  Question: \"{question}\"")
        print(f"  A) {row['answerA']}")
        print(f"  B) {row['answerB']}")
        print(f"  C) {row['answerC']}")
        print(f"  Ground truth: {ground_truth}")

        prompt_text = build_prompt(
            context, question,
            row["answerA"], row["answerB"], row["answerC"],
        )

        with kbench.chats.new(f"sample_{i}"):
            try:
                raw_response = llm.prompt(prompt_text)
            except Exception as e:
                print(f"  [ERROR] LLM call failed: {e}")
                raw_response = None

        predicted = normalize_answer(raw_response)

        correct = predicted == ground_truth
        print(f"  Predicted: {predicted} (raw: '{str(raw_response)[:40]}...')")
        print(f"  {'CORRECT' if correct else 'WRONG'}")

        y_true.append(ground_truth)
        y_pred.append(predicted)
        results_log.append({
            "sample": i + 1,
            "context": display_ctx,
            "ground_truth": ground_truth,
            "predicted": predicted,
            "correct": correct,
        })

    # STEP 4: Compute Weighted-Average F1 Score
    all_labels = VALID_ANSWERS + (["INVALID"] if "INVALID" in y_pred else [])

    waf1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        labels=all_labels,
        zero_division=0,
    )

    # STEP 5: Summary
    correct_count = sum(r["correct"] for r in results_log)
    invalid_count = sum(1 for p in y_pred if p == "INVALID")

    print(f"\n{'='*60}")
    print("RESULTS SUMMARY")
    print(f"{'='*60}")

    results_df = pd.DataFrame(results_log)
    print(results_df[["sample", "ground_truth", "predicted", "correct"]].to_string(
        index=False
    ))

    print(f"\nAccuracy:  {correct_count}/{total} ({correct_count/total:.1%})")
    print(f"Invalid:   {invalid_count}/{total}")
    print(f"WAF1:      {waf1:.4f}")

    if total >= 3:
        print(f"\nPer-class report:")
        print(classification_report(
            y_true, y_pred, labels=VALID_ANSWERS, zero_division=0
        ))

    print(f"{'='*60}")

    # STEP 6: Assertions
    kbench.assertions.assert_true(
        0.0 <= waf1 <= 1.0,
        expectation="Weighted-Average F1 score should be between 0 and 1."
    )

    kbench.assertions.assert_true(
        invalid_count < total,
        expectation="At least one prediction should be a valid answer label."
    )

    kbench.assertions.assert_true(
        correct_count > 0,
        expectation="Model should correctly answer at least one question."
    )

    return waf1


# ============================================================================
# RUN THE TASK
# ============================================================================

social_iqa_reasoning.run(kbench.llm)

[INFO] Loaded 1954 valid samples from /kaggle/input/datasets/thedevastator/social-i-qa-a-dataset-for-social-inquiry-questio/validation.csv
[INFO] Answer distribution:
answer
A    643
B    654
C    657
[INFO] Selected 250 samples (seed=42)
[INFO] Selection distribution:
answer
A    87
B    80
C    83

[SAMPLE 1/250]
  Context:  "Carson kissed Alex gently and asked if she wanted to dance at the club..."
  Question: "What will Alex want to do next?"
  A) skilled
  B) lazy
  C) needs ask Alex do dance
  Ground truth: A
  Predicted: C (raw: 'C...')
  WRONG

[SAMPLE 2/250]
  Context:  "Kai gave Ash some bread and stuff from the deli to make a sandwich."
  Question: "How would Ash feel as a result?"
  A) like they were nice
  B) Grateful for the trade
  C) Glad to be able to make a sandwich
  Ground truth: C
  Predicted: C (raw: 'C...')
  CORRECT

[SAMPLE 3/250]
  Context:  "Kai had got into bed and felt relaxed."
  Question: "How would you describe Kai?"
  A) feeling tired
  B) wanting to le